# Spanish MT — mBART Benchmark

Evaluation of `facebook/mbart-large-50-many-to-many-mmt` on Spanish→English translation.  
Datasets: Europarl Corpus (primary), OPUS MT Test (secondary).  

See [`README.md`](README.md) for evaluation notes and results.  

**Setup:** run the install cell. Set runtime to GPU (T4/V100/A100) for production-scale runs.

## Setup

**Google Colab:** run the install cell.  
**Local:** `pip install transformers sentencepiece bert_score sacrebleu nltk accelerate`

In [ ]:
%%capture
!pip install transformers sentencepiece bert_score sacrebleu nltk accelerate

import nltk
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
import subprocess, datetime as dt
import torch, numpy as np, pandas as pd
from tqdm.notebook import tqdm
from bert_score import score as bert_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.data[idx], return_tensors="pt",
            padding="max_length", truncation=True, max_length=self.max_length
        )
        return {k: v.squeeze() for k, v in enc.items()}

## Load datasets

Europarl: https://www.statmt.org/europarl/  
OPUS MT test: https://huggingface.co/Helsinki-NLP/opus-mt-es-en

Both should be CSVs with `source` (Spanish) and `target` (English gold standard) columns.

In [ ]:
# Load evaluation datasets
# Europarl: https://www.statmt.org/europarl/  — columns: source (es), target (en)
# OPUS MT test: https://huggingface.co/Helsinki-NLP/opus-mt-es-en (evaluation split)
# Both should be CSVs with 'source' and 'target' columns

europarl = pd.read_csv('evaluation_data/europarl_corpus.csv')
opus     = pd.read_csv('evaluation_data/opus_corpus.csv')

dataset_1 = europarl.sample(1000, random_state=1)  # Europarl — primary benchmark
dataset_2 = opus.sample(1000, random_state=1)       # OPUS — secondary

dataset      = dataset_1
dataset_name = 'europarl'

assert 'source' in dataset.columns and 'target' in dataset.columns
print(f'{dataset_name}: {len(dataset)} rows')
dataset.head()

## Translate — facebook/mbart-large-50-many-to-many-mmt

Change `dataset` and `dataset_name` at the top to switch between Europarl and OPUS.

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

current_model = 'facebook/mbart-large-50-many-to-many-mmt'
src_lang = 'es_XX'  # <<<< source language code — see model card for all codes

model     = MBartForConditionalGeneration.from_pretrained(current_model)
tokenizer = MBart50TokenizerFast.from_pretrained(current_model)
tokenizer.src_lang = src_lang

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'Running on: {device}')

ds = TranslationDataset(dataset['source'].tolist(), tokenizer)
dl = DataLoader(ds, batch_size=16, shuffle=False)

translations, diffs = [], []
last = dt.datetime.today().timestamp()
start = last

for batch in tqdm(dl):
    data = {k: v.to(device) for k, v in batch.items()}
    tokens = model.generate(**data, forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"])
    translations.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True))
    now = dt.datetime.today().timestamp()
    diffs.append(now - last); last = now

duration = round(dt.datetime.today().timestamp() - start, 2)
print(f"Dataset: {dataset_name} ({len(dataset)} samples)  |  Duration: {duration}s  |  Avg: {round(np.mean(diffs),2)}s/it  |  Device: {device}")
dataset['machine_translation'] = translations
dataset.head()

## Evaluate — METEOR

In [ ]:
import nltk
meteor = nltk.translate.meteor_score.meteor_score
scores = []
for ref, hyp in tqdm(zip(dataset['target'], dataset['machine_translation'])):
    scores.append(meteor([ref.split()], hyp.split()))
dataset['meteor_score'] = scores
print(f"Average METEOR: {np.mean(scores):.4f}")

## Evaluate — BERTScore

In [ ]:
references = dataset['target'].tolist()
cands      = dataset['machine_translation'].tolist()

tok = AutoTokenizer.from_pretrained('roberta-large')
refs_tok  = [" ".join(tok.tokenize(r)) for r in references]
cands_tok = [" ".join(tok.tokenize(c)) for c in cands]

P, R, F1 = bert_score(cands_tok, refs_tok, model_type='roberta-large', lang='en', verbose=True)
dataset['bert_score'] = F1.numpy()
print(f"Average BERTScore F1: {F1.mean():.4f}")

## Evaluate — mBERTScore

Multilingual BERTScore — compares source and translation without requiring a gold standard.

In [ ]:
tok = AutoTokenizer.from_pretrained('microsoft/deberta-large-mnli')
refs_tok  = [" ".join(tok.tokenize(r)) for r in dataset['target'].tolist()]
cands_tok = [" ".join(tok.tokenize(c)) for c in dataset['machine_translation'].tolist()]

P, R, F1 = bert_score(cands_tok, refs_tok, model_type='microsoft/deberta-large-mnli', verbose=True)
dataset['bilingual_score'] = F1.numpy()
print(f"Average mBERTScore F1: {F1.mean():.4f}")

## Summary & save outputs

In [ ]:
results = dataset[['meteor_score','bert_score','bilingual_score']].describe()
print(current_model, '—', dataset_name)
display(results)

# Save outputs
import os
out_dir = f'evaluation_output/{dataset_name}'
os.makedirs(out_dir, exist_ok=True)
dataset.to_csv(f'{out_dir}/mt_output.csv', index=False)
dataset.sample(n=min(100, len(dataset)), random_state=1).to_csv(f'{out_dir}/mt_samples.csv', index=False)
results.to_csv(f'{out_dir}/evaluation_results.csv')
print(f'Saved to {out_dir}/')

## Tokenisation check

Flag sentences that will be truncated by the model's max sequence length.

In [ ]:
def check_tokenisation(tokenizer, sentences):
    rows = []
    for s in tqdm(sentences):
        tokens = tokenizer.tokenize(s)
        n = len(tokens)
        rows.append({"sentence": s, "num_tokens": n,
                     "truncated": n > tokenizer.model_max_length,
                     "truncation_amount": max(0, n - tokenizer.model_max_length)})
    return pd.DataFrame(rows)

check_tokenisation(tokenizer, dataset['source'].tolist()).head(20)